# Inter-Annotator Agreement (Fleiss' Kappa)

This notebo computes Fleiss' kappa across the three annotators


In [2]:
import pandas as pd

try:
    from statsmodels.stats.inter_rater import fleiss_kappa
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "statsmodels is not installed in this Jupyter kernel. "
        "Run `%pip install statsmodels openpyxl` in a notebook cell, then restart the kernel."
    ) from e

file_path = r"manual-labelling.xlsx"
annotator_cols = ["user1", "user2", "user3"]

try:
    df = pd.read_excel(file_path)
except ImportError as e:
    raise ImportError(
        "openpyxl is required to read .xlsx files. "
        "Run `%pip install openpyxl` in a notebook cell, then restart the kernel."
    ) from e

missing = [c for c in annotator_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing annotator columns: {missing}")

for c in annotator_cols:
    df[c] = df[c].astype("string").str.strip()

clean = df.dropna(subset=annotator_cols).copy()
labels = sorted(pd.unique(clean[annotator_cols].values.ravel("K")))

counts = pd.DataFrame(0, index=clean.index, columns=labels, dtype=int)
for col in annotator_cols:
    one_hot = pd.get_dummies(clean[col]).reindex(columns=labels, fill_value=0)
    counts = counts.add(one_hot, fill_value=0).astype(int)

kappa = fleiss_kappa(counts.values, method="fleiss")

print(f"Total rows in workbook: {len(df)}")
print(f"Rows used for agreement (all 3 annotators present): {len(clean)}")
print(f"Labels found: {labels}")
print(f"Fleiss' kappa: {kappa:.6f}")

counts.head()

Total rows in workbook: 1019
Rows used for agreement (all 3 annotators present): 1019
Labels found: ['Negative', 'Neutral', 'Positive']
Fleiss' kappa: 0.941668


,Negative,Neutral,Positive
0,0,0,3
1,0,0,3
2,0,0,3
3,3,0,0
4,1,0,2


In [3]:
# Optional: label distribution per annotator
for col in annotator_cols:
    print(f"\n{col} label counts:")
    print(clean[col].value_counts())


user1 label counts:
user1
Positive    445
Neutral     314
Negative    260
Name: count, dtype: Int64

user2 label counts:
user2
Positive    435
Neutral     327
Negative    257
Name: count, dtype: Int64

user3 label counts:
user3
Positive    441
Neutral     311
Negative    267
Name: count, dtype: Int64


## Result from this dataset
Using all 1,019 rows and the three annotators (`user1`, `user2`, `user3`), Fleiss' kappa is approximately **0.941668**, indicating very high agreement.
